<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/06%20-%20Quantificadores%20e%20Predicados%20em%20Redes%20de%20Sensores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 06 - Notebook: Motor de Varredura de Predicados em Redes de Sensores

Este notebook implementa os quantificadores universais ($\forall$ - FORALL) e existenciais ($\exists$ - EXISTS) da Lógica de Primeira Ordem (FOL) sobre a malha de instrumentos do AGV, permitindo validar a integridade total do sistema ou disparar interrupções de emergência em caso de falha.

In [1]:
from typing import Dict, List, Callable
import pandas as pd

# Funções Operadoras dos Quantificadores de Lógica de Primeira Ordem (FOL)

def FORALL(dominio: List[dict], predicado: Callable[[dict], bool]) -> bool:
    """Retorna True se e somente se o predicado for verdadeiro para TODOS os elementos."""
    return all(predicado(elemento) for elemento in dominio)

def EXISTS(dominio: List[dict], predicado: Callable[[dict], bool]) -> bool:
    """Retorna True se o predicado for verdadeiro para AO MENOS UM elemento."""
    return any(predicado(elemento) for elemento in dominio)

print("Motor de quantificadores (FORALL / EXISTS) carregado com sucesso.")

Motor de quantificadores (FORALL / EXISTS) carregado com sucesso.


## 1. Mapeamento do Universo de Sensores do AGV ($\mathcal{S}_{AGV}$)

Definição dos instrumentos de campo embarcados e suas funções de validação de estado normal.

In [2]:
# Universo de discurso: Lista de sensores do AGV
sensores_agv = [
    {"tag": "ESD-200", "tipo": "E-Stop", "status_falha": False},
    {"tag": "AT-201",  "tipo": "Gás NH3", "status_falha": False},
    {"tag": "BT-201",  "tipo": "Bateria", "status_falha": False},
    {"tag": "CAM-102", "tipo": "Colisão", "status_falha": False}
]

# Predicado P(x): Define se o sensor x está em estado de FALHA
def esta_em_falha(sensor: dict) -> bool:
    return sensor["status_falha"] == True

# Evaluador da Malha usando Quantificadores
def avaliar_seguranca_malha(malha: List[dict]) -> Dict[str, bool]:
    # Permissivo: TODOS os sensores devem estar SEM falha (NOT esta_em_falha)
    permissivo_forall = FORALL(malha, lambda s: not esta_em_falha(s))

    # Trip: EXISTE ao menos UM sensor em falha
    trip_exists = EXISTS(malha, esta_em_falha)

    return {
        "Permissivo_Movimento": permissivo_forall,
        "Trip_Emergencia": trip_exists
    }

print("Resultado Inicial (Condição Normal de Operação):")
print(avaliar_seguranca_malha(sensores_agv))

Resultado Inicial (Condição Normal de Operação):
{'Permissivo_Movimento': True, 'Trip_Emergencia': False}


## 2. Injeção Dinâmica de Falhas na Malha

Simulação de diferentes cenários industriais injetando falhas distribuídas nos sensores para validar a resposta do SCADA-Core.

In [3]:
cenarios_teste = [
    {"nome": "Operação 100% Normal", "falhas": []},
    {"nome": "Vazamento de Amônia (AT-201)", "falhas": ["AT-201"]},
    {"nome": "Parada de Emergência Manual (ESD-200)", "falhas": ["ESD-200"]},
    {"nome": "Múltiplas Falhas (Bateria + Colisão)", "falhas": ["BT-201", "CAM-102"]}
]

relatorio = []

for c in cenarios_teste:
    # Reset da malha
    malha_simulada = [
        {"tag": "ESD-200", "status_falha": "ESD-200" in c["falhas"]},
        {"tag": "AT-201",  "status_falha": "AT-201" in c["falhas"]},
        {"tag": "BT-201",  "status_falha": "BT-201" in c["falhas"]},
        {"tag": "CAM-102", "status_falha": "CAM-102" in c["falhas"]}
    ]

    res = avaliar_seguranca_malha(malha_simulada)

    relatorio.append({
        "Cenário": c["nome"],
        "FORALL (Permissivo)": res["Permissivo_Movimento"],
        "EXISTS (Trip)": res["Trip_Emergencia"],
        "Estado do AGV": "AVANÇANDO" if res["Permissivo_Movimento"] else "PARADA DE EMERGÊNCIA"
    })

pd.DataFrame(relatorio)

,Cenário,FORALL (Permissivo),EXISTS (Trip),Estado do AGV
0,Operação 100% Normal,True,False,AVANÇANDO
1,Vazamento de Amônia (AT-201),False,True,PARADA DE EMERGÊNCIA
2,Parada de Emergência Manual (ESD-200),False,True,PARADA DE EMERGÊNCIA
3,Múltiplas Falhas (Bateria + Colisão),False,True,PARADA DE EMERGÊNCIA
